# ConnectTel – Predictive Customer Churn Analysis

End-to-end churn analysis, feature engineering, model comparison, evaluation and explainability.

**Dataset:** IBM Telco Customer Churn (Kaggle). Place the CSV in `data/raw/` before running.

## 1. Business problem and objectives
Predict customers at risk of cancelling service so the retention team can prioritize interventions. The project emphasizes ROC-AUC, recall, precision, explainability and business usefulness rather than accuracy alone.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_data
from src.data_cleaning import clean_data
from src.feature_engineering import add_features


In [ ]:
raw = load_data()
df = clean_data(raw)
print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)


## 2. Data quality and missing values
TotalCharges is converted to numeric. Blank values are handled using monthly charge multiplied by tenure; this preserves zero-tenure customers.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])
print('Duplicates:', df.duplicated().sum())
print(df['Churn'].value_counts(normalize=True).rename('proportion'))


## 3. Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x='Churn', ax=axes[0])
sns.histplot(data=df, x='tenure', hue='Churn', kde=True, ax=axes[1])
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, ax=axes[2])
for ax in axes: ax.set_title(ax.get_title() or 'Customer distribution')
plt.tight_layout(); plt.show()


## 4. Bivariate analysis

In [ ]:
for col in ['Contract','InternetService','PaymentMethod','Partner','TechSupport','OnlineSecurity']:
    rates = df.groupby(col, dropna=False)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
    ax = rates.plot(kind='bar', figsize=(8,3), title=f'Churn rate by {col}', ylabel='Churn rate')
    ax.set_ylim(0,1); plt.tight_layout(); plt.show()


## 5. Correlation and hypothesis testing
Chi-square tests evaluate categorical associations. Mann–Whitney U tests compare numerical distributions without assuming normality. Statistical association does not prove causation.

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu
for col in ['InternetService','Partner','Contract']:
    tab = pd.crosstab(df[col], df['Churn'])
    chi2, p, dof, expected = chi2_contingency(tab)
    print(f'{col}: chi2={chi2:.3f}, p={p:.6g}')
churn_tenure = df.loc[df.Churn=='Yes','tenure']
stay_tenure = df.loc[df.Churn=='No','tenure']
print('Tenure Mann-Whitney:', mannwhitneyu(churn_tenure, stay_tenure, alternative='two-sided'))


## 6. Feature engineering

In [ ]:
featured = add_features(df)
print(featured[['TotalChargesPerTenure','ServiceCount','TenureGroup','IsNewCustomer','IsMonthToMonth','HasInternetService']].head())


## 7. Model training, cross-validation and tuning
The training script uses a stratified 80/20 split, preprocessing inside each pipeline, 5-fold stratified CV and randomized hyperparameter search. Run it from the project root.

In [ ]:
from src.model_training import train_all
metadata, fitted_models, test_data = train_all(search_iterations=12)
print('Best model:', metadata['best_model'])
display(pd.DataFrame(metadata['test_results']).T)


## 8. Confusion matrices and ROC curves

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
X_test, y_test = test_data
fig, axes = plt.subplots(1, len(fitted_models), figsize=(5*len(fitted_models),4))
if len(fitted_models)==1: axes=[axes]
for ax, (name, model) in zip(axes, fitted_models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,6))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)
plt.plot([0,1],[0,1],'--',label='Random baseline'); plt.legend(); plt.show()


## 9. SHAP explainability
SHAP can be computationally expensive for a full pipeline. A representative sample is used. The explainer operates on the selected model's probability function, so the displayed explanations are model-specific.

In [ ]:
import shap
best = fitted_models[metadata['best_model']]
X_sample = X_test.sample(min(100, len(X_test)), random_state=42)
explainer = shap.Explainer(best.predict_proba, X_sample)
shap_values = explainer(X_sample)
# For binary classifiers, select the positive-class output when SHAP returns class outputs.
sv = shap_values[..., 1] if len(shap_values.shape) == 3 else shap_values
shap.plots.beeswarm(sv, max_display=15)
shap.plots.bar(sv, max_display=15)


## 10. Business insights and recommendations
Use the computed model results and SHAP plots to identify risk drivers. Test retention offers with controlled experiments before claiming causal impact or revenue savings.

**Limitations:** This is a historical public dataset. It does not include real-time customer behavior, current competitor offers or verified intervention outcomes.